In [1]:
import pandas as pd
import numpy as np
import warnings
import re
import torch
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, classification_report
from torch.optim import Adam
import torch.nn as nn
import torch.nn.functional as F
from torch import optim

warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv(r'reddit_dataset.csv')
df.drop(columns = ['row_id','subreddit'], inplace=True)

In [3]:
y_labels_original = df[['rule_violation']]
x = df.drop(columns = ['rule_violation'])

In [4]:
def cleaning_text(text_file):

     ## Removing URLs
    pattern = re.compile('https?:\/\/\S+|www\.\S+|Https?:\/\/\S+|\S+\.com\S+|\S+\.com|\[.*?\]|\S+ \. com.*')
    for i in range(len(text_file)):
        text_file[i] = pattern.sub(r'',text_file[i])

    ##Removing HTML rags
    pattern = re.compile('<.*?>')
    for i in range(len(text_file)):
        text_file[i] = pattern.sub(r'',text_file[i])

    ## Removing Emails and Hashtags
    pattern = re.compile('#\S+|@\S+|\S+\@\S+|\S+@')
    for i in range(len(text_file)):
        text_file[i] = pattern.sub(r'',text_file[i])

    ### Removing username and subreddit mentions
    pattern = re.compile('u\/\S+|r\/\S+')
    for i in range(len(text_file)):
        text_file[i] = pattern.sub(r'',text_file[i])

    #emotions, symbols, pictographs, transport and map symbols, flags etx.
    pattern = re.compile("["
                            u"\U0001F600-\U0001F64F"  # emoticons
                            u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                            u"\U0001F680-\U0001F6FF"  # transport & map symbols
                            u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                            u"\U00002702-\U000027B0"
                            u"\U000024C2-\U0001F251"
                            "]+", flags=re.UNICODE)
    for i in range(len(text_file)):
        text_file[i] = pattern.sub(r'',text_file[i])

    ##Removing Numbers & \n spaces
    pattern = re.compile('\d|\\n')
    for i in range(len(text_file)):
        text_file[i] = pattern.sub(r'',text_file[i])

    return text_file

##TRAINING DATA ----------------------
x['body'] = cleaning_text(list(x['body']))
x['positive_example_1'] = cleaning_text(list(x['positive_example_1']))
x['positive_example_2'] = cleaning_text(list(x['positive_example_2']))
x['negative_example_1'] = cleaning_text(list(x['negative_example_1']))
x['negative_example_2'] = cleaning_text(list(x['negative_example_2']))

### Tokenization of text



In [5]:
x.shape

(2029, 6)

In [6]:
x.head()

,body,rule,positive_example_1,positive_example_2,negative_example_1,negative_example_2
0,Banks don't want you to know this! Click here ...,"No Advertising: Spam, referral links, unsolici...",If you could tell your younger self something ...,hunt for lady for jack off in neighbourhood,Watch Golden Globe Awards Live Online in HD C...,"DOUBLE CEE x BANDS EPPS - ""BIRDS""\r\rDOWNLOAD/..."
1,SD Stream,"No Advertising: Spam, referral links, unsolici...",(,LOLGA.COM is One of the First Professional Onl...,\rStraight Outta Cross Keys SC YouTube Search...,No one would argue the fact that Google is o...
2,Lol. Try appealing the ban and say you won't d...,No legal advice: Do not offer or request legal...,Don't break up with him or call the cops. If ...,It'll be dismissed: \r\rThe first amendment la...,Where is there a site that still works where y...,Because this statement of his is true. It isn'...
3,she will come your home open her legs with an...,"No Advertising: Spam, referral links, unsolici...",Selling Tyrande codes for € to paypal. PM. \r,tight pussy watch for your cock get her at thi...,NSFW(obviously),Good News ::Download WhatsApp .. APK for Andro...
4,code free tyrande --->>> \r\rfor you and your ...,"No Advertising: Spam, referral links, unsolici...",wow!! amazing reminds me of the old days.Well...,seek for lady for sex in around,must be watch movie,We're streaming Pokemon Veitnamese Crystal RIG...


In [7]:
x_positive_1_df = x[['rule', 'positive_example_1']].rename(columns={'positive_example_1': 'body'})
x_positive_2_df = x[['rule', 'positive_example_2']].rename(columns={'positive_example_2': 'body'})
x_negative_1_df = x[['rule', 'negative_example_1']].rename(columns={'negative_example_1': 'body'})
x_negative_2_df = x[['rule', 'negative_example_2']].rename(columns={'negative_example_2': 'body'})
x_body_df = x[['rule','body']]

In [8]:
x_positive_1_df['rule_violation'] = 1
x_positive_2_df['rule_violation'] = 1
x_negative_1_df['rule_violation'] = 0
x_negative_2_df['rule_violation'] = 0
x_body_df['rule_violation'] = y_labels_original['rule_violation']

In [9]:
df = pd.concat([x_positive_1_df, x_positive_2_df, x_negative_1_df, x_negative_2_df, x_body_df], axis = 0).reset_index(drop=True)

In [10]:
df.head()

,rule,body,rule_violation
0,"No Advertising: Spam, referral links, unsolici...",If you could tell your younger self something ...,1
1,"No Advertising: Spam, referral links, unsolici...",(,1
2,No legal advice: Do not offer or request legal...,Don't break up with him or call the cops. If ...,1
3,"No Advertising: Spam, referral links, unsolici...",Selling Tyrande codes for € to paypal. PM. \r,1
4,"No Advertising: Spam, referral links, unsolici...",wow!! amazing reminds me of the old days.Well...,1


In [11]:
rule_text_list = df['rule'].to_list()
body_text_list = df['body'].to_list()

def list_to_str(list_file: list):
  return ' '.join(map(str, list_file))

rule_text = list_to_str(rule_text_list)
body_text = list_to_str(body_text_list)

In [12]:
import spacy

eng_lan  = spacy.blank("en")

eng_lan.max_length = 10000000

In [13]:
doc = eng_lan(rule_text)
rule_worded_out = [i.text for i in doc]

doc = eng_lan(body_text)
body_worded_out = [i.text for i in doc]

In [14]:
body_vocab = list(set(body_worded_out))
rule_vocab = list(set(rule_worded_out))

body_vocab_size = len(body_vocab)
rule_vocab_size = len(rule_vocab)

In [15]:
body_vocab_size,rule_vocab_size

(8130, 22)

In [16]:
with open('body_text','w', encoding = 'utf-8') as f:
  for line in body_text_list:
    f.write(line.strip() + '\n')

with open('rule_text','w', encoding = 'utf-8') as f:
  for line in rule_text_list:
    f.write(line.strip() + "\n")

In [17]:
import sentencepiece as spm

In [ ]:
spm.SentencePieceTrainer.train(
    input = '/content/rule_text',
    model_prefix = 'rule_tokenizer',
    vocab_size = 290,
    character_coverage = 1,
    model_type = 'bpe',
    control_symbols=["<pad>", "<sos>", "<eos>"],
    hard_vocab_limit=False,
    byte_fallback = True
)


In [19]:
spm.SentencePieceTrainer.train(
    input = '/content/body_text',
    model_prefix = 'body_tokenizer',
    vocab_size = 10000,
    character_coverage = 1,
    model_type = 'bpe',
    control_symbols=["<pad>", "<sos>", "<eos>"],
    hard_vocab_limit=False,
    byte_fallback = True
)

Now Tokenization of the lists

In [20]:
body_tokenizer = spm.SentencePieceProcessor()
body_tokenizer.load('/content/body_tokenizer.model')
pad_id = body_tokenizer.piece_to_id("<pad>")#same in both lang.

rule_tokenizer = spm.SentencePieceProcessor()
rule_tokenizer.load('/content/rule_tokenizer.model')

True

In [21]:
tokenized_stored = [body_tokenizer.encode(i,add_bos = True, add_eos = True , out_type = int)
                    for i in body_text_list]
max_len_body = max(len(x) for x in tokenized_stored)
print(max_len_body)
body_text_tokenized = [x + [pad_id] * (max_len_body - len(x)) for x in tokenized_stored]


181


In [22]:
tokenized_stored = [rule_tokenizer.encode(i, add_bos = True, add_eos = True ,out_type = int)
                  for i in rule_text_list]
max_len_rule = max(len(x) for x in tokenized_stored)
print(max_len_rule)
rule_text_tokenized = [x + [pad_id] * (max_len_rule - len(x)) for x in tokenized_stored]

106


#### Rule will look same because they are just 2 unique entries

In [23]:
sequence_length = max_len_rule + max_len_body

In [24]:
df['rule'] = rule_text_tokenized
df['body'] = body_text_tokenized

### Now let's put it in torch datasets

In [25]:
df.columns

Index(['rule', 'body', 'rule_violation'], dtype='object')

In [26]:
from sklearn.model_selection import train_test_split

x = df.drop(columns = ['rule_violation'])
y = df[['rule_violation']]

x_train,x_test,y_train,y_test = train_test_split(x,y, train_size = 0.75, random_state = 42)

body_train = x_train["body"].tolist()
rule_train = x_train["rule"].tolist()

body_test  = x_test["body"].tolist()
rule_test  = x_test["rule"].tolist()

In [27]:
import numpy as np

body_train_np = np.stack(body_train).astype(np.int64)
rule_train_np = np.stack(rule_train).astype(np.int64)
body_test_np  = np.stack(body_test).astype(np.int64)
rule_test_np  = np.stack(rule_test).astype(np.int64)


# NumPy → Torch
body_train = torch.from_numpy(body_train_np)
rule_train = torch.from_numpy(rule_train_np)
body_test  = torch.from_numpy(body_test_np)
rule_test  = torch.from_numpy(rule_test_np)


y_train_np = np.asarray(y_train, dtype=np.int64)
y_test_np  = np.asarray(y_test, dtype=np.int64)
y_train = torch.from_numpy(y_train_np)
y_test  = torch.from_numpy(y_test_np)

In [28]:
y_train.shape, y_train.view(-1).shape


(torch.Size([7608, 1]), torch.Size([7608]))

In [29]:
from torch.utils.data import Dataset, DataLoader

In [30]:
class CustomDataset(Dataset):
  def __init__(self, body_text, rule_text, labels):
    super().__init__()

    self.body_text = body_text
    self.rule_text = rule_text
    self.labels = labels

  def __len__(self):
    return  len(self.labels)

  def __getitem__(self, idx):
    return  self.body_text[idx],self.rule_text[idx],self.labels[idx]



In [ ]:

train_data = CustomDataset(body_train, rule_train,y_train)
test_data = CustomDataset(body_test, rule_test,y_test)


In [32]:
train_loader = DataLoader(train_data, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_data, batch_size = 32, shuffle = True)

In [33]:

class ViolationClassifier(nn.Module):
    def __init__(self,
                 num_embedding=10000,
                 embedding_dim=768,
                 hidden_dim=256,
                 dropout=0.2, pad_id = pad_id):
        super(ViolationClassifier, self).__init__()

        # First fully connected layer

        self.embed1 = nn.Embedding(num_embeddings=num_embedding,
                                   embedding_dim=embedding_dim,
                                   padding_idx = pad_id)

        self.fc1 = nn.Linear(embedding_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.dropout1 = nn.Dropout(dropout)

        # Second fully connected layer
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.bn2 = nn.BatchNorm1d(hidden_dim // 2)
        self.dropout2 = nn.Dropout(dropout)

        # Output layer
        self.fc_out = nn.Linear(hidden_dim // 2, 2)  # Binary classification output

    def forward(self, rule_text, body_text):  # Concatenate embeddings

        x = torch.cat([rule_text, body_text], dim = 1)

        x = self.embed1(x)

        x = x.mean(dim=1)

        x = self.fc1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.dropout2(x)

        out = self.fc_out(x)
        return out


In [34]:

# Check if CUDA is available and use GPU if it is
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ViolationClassifier().to(device) # Initialize with new parameters

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 25 # Define number of epochs



In [36]:
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for body_batch, rule_batch, y_batch in train_loader:
        # Move data to the same device as the model
        body_batch, rule_batch, y_batch = body_batch.to(device), rule_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(body_batch, rule_batch)
        loss = criterion(outputs, y_batch.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()


In [53]:
model.eval()
y_pred, y_actual = [], []
with torch.no_grad():
    for body_batch, rule_batch, y_batch in test_loader:
        # Move data to the same device as the model
        body_batch, rule_batch, y_batch = body_batch.to(device), rule_batch.to(device), y_batch.to(device)

        outputs = model(body_batch, rule_batch)
        preds = torch.argmax(outputs, dim=1)
        y_pred.extend(preds.cpu())
        y_actual.extend(y_batch.cpu())



In [54]:
type(y_pred),type(y_actual)

(list, list)

In [55]:
y_actual

[tensor([1]),
 tensor([0]),
 tensor([0]),
 tensor([1]),
 tensor([1]),
 tensor([1]),
 tensor([1]),
 tensor([0]),
 tensor([1]),
 tensor([0]),
 tensor([1]),
 tensor([1]),
 tensor([1]),
 tensor([1]),
 tensor([1]),
 tensor([1]),
 tensor([1]),
 tensor([1]),
 tensor([0]),
 tensor([0]),
 tensor([1]),
 tensor([0]),
 tensor([1]),
 tensor([0]),
 tensor([1]),
 tensor([0]),
 tensor([0]),
 tensor([1]),
 tensor([0]),
 tensor([1]),
 tensor([0]),
 tensor([0]),
 tensor([1]),
 tensor([0]),
 tensor([0]),
 tensor([0]),
 tensor([1]),
 tensor([1]),
 tensor([0]),
 tensor([1]),
 tensor([0]),
 tensor([0]),
 tensor([1]),
 tensor([1]),
 tensor([1]),
 tensor([0]),
 tensor([0]),
 tensor([1]),
 tensor([1]),
 tensor([0]),
 tensor([1]),
 tensor([0]),
 tensor([0]),
 tensor([1]),
 tensor([1]),
 tensor([1]),
 tensor([1]),
 tensor([1]),
 tensor([0]),
 tensor([1]),
 tensor([0]),
 tensor([1]),
 tensor([1]),
 tensor([1]),
 tensor([1]),
 tensor([0]),
 tensor([1]),
 tensor([1]),
 tensor([1]),
 tensor([0]),
 tensor([0]),
 tenso

In [56]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

def print_classification_metrics(y_true, y_pred, y_proba=None):
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision:  {precision_score(y_true, y_pred):.4f}")
    print(f"Recall :     {recall_score(y_true, y_pred):.4f}")
    print(f"F1-score:   {f1_score(y_true, y_pred):.4f}")
    print(f"roc_auc_score:   {roc_auc_score(y_true, y_pred):.4f}")



In [57]:
print_classification_metrics(np.array(y_actual),np.array(y_pred))

Accuracy: 0.9417
Precision:  0.9618
Recall :     0.9214
F1-score:   0.9412
roc_auc_score:   0.9419


#### Had problems with shape of target, had to fix that.


In [ ]:
with torch.no_grad():
  pred_data = []
  actual_data = []
  for body_batch, rule_batch, y_batch in train_loader:
    body_batch, rule_batch, y_batch = body_batch.to(device), rule_batch.to(device), y_batch.to(device)
    outputs = model(body_batch, rule_batch)
    outputs = torch.argmax(outputs, dim=1)
    pred_data.extend(outputs.cpu())
    actual_data.extend(y_batch.cpu())

y_pred = np.array(pred_data).reshape(-1,1)
y_actual = np.array(actual_data).reshape(-1,1)

print_classification_metrics(y_true = y_actual, y_pred = y_pred)

In [ ]:
with torch.no_grad():
  pred_data = []
  actual_data = []
  for body_batch, rule_batch, y_batch in val_loader:
    body_batch, rule_batch, y_batch = body_batch.to(device), rule_batch.to(device), y_batch.to(device)
    outputs = model(body_batch, rule_batch)
    outputs = torch.argmax(outputs, dim=1)
    pred_data.extend(outputs.cpu())
    actual_data.extend(y_batch.cpu())

y_pred = np.array(pred_data).reshape(-1,1)
y_actual = np.array(actual_data).reshape(-1,1)

print_classification_metrics(y_true = y_actual, y_pred = y_pred)

In [ ]:
torch.save(model.state_dict(), "model.pth")

# Newer Dataset

In [ ]:
text_name = tokenizer(text_name, padding=True, truncation=True, return_tensors='pt')
text_name = text_name.to("cuda")

In [ ]:
embeddings_list = []
batch_size = 32
for i in range(0, len(text_name['input_ids']), batch_size):
  tokenized_text = {k: v[i:i+batch_size] for k,v in text_name.items()}
  with torch.no_grad():
    text_tokenized = model(**tokenized_text)
  embeddings_list.append(text_tokenized.last_hidden_state[:,0,:])
  concatinated_text = torch.cat(embeddings_list, dim = 0)
  df_col = [emb.tolist() for emb in concatinated_text]


## Text is properly embedded

In [ ]:
df['text'] = df_col

In [ ]:
# x_train = x_train.squeeze().astype(int)
# x_test = x_test.squeeze().astype(int)

In [ ]:
from sklearn.model_selection import train_test_split

y = df[['target']]
x = df[['text']]

x_train,x_test,y_train,y_test = train_test_split(x,y, test_size = 0.25, random_state = 42)
y_train['target'] = y_train['target'].apply(lambda x:0 if x == 'ham' else 1)
y_test['target'] = y_test['target'].apply(lambda x:0 if x == 'ham' else 1)

In [ ]:
x_train = [item[0] for item in np.array(x_train)]
y_train = np.array(y_train)

x_test = [item[0] for item in np.array(x_test)]
y_test = np.array(y_test)

In [ ]:

x_train = torch.tensor(x_train, dtype = torch.float32).squeeze()
y_train = torch.tensor(y_train, dtype = torch.long).view(-1,1)
x_test = torch.tensor(x_test, dtype = torch.float32).squeeze()
y_test = torch.tensor(y_test, dtype = torch.long).view(-1,1)

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim

In [ ]:
class dataclass(Dataset):
  def __init__(self, text_tensors, target_tensors):
    self.text = text_tensors
    self.target = target_tensors

  def __len__(self):
    return len(self.text)

  def __getitem__(self, idx):
    return self.text[idx], self.target[idx]

In [ ]:
train_data = dataclass(x_train, y_train)
test_data = dataclass(x_test, y_test)

train_data = DataLoader(train_data, batch_size = 32, shuffle=True)
test_data = DataLoader(test_data, batch_size = 32)

In [ ]:
import torch

In [ ]:
class MyNN(nn.Module):
  def __init__(self, hidden_dim=128 ,input_dim=768,drop_rate = 0.2):

    super().__init__()

    self.linear1 = nn.Linear(input_dim, hidden_dim) # Corrected input_dim for single embedding
    self.bn1 = nn.BatchNorm1d(hidden_dim)
    self.drop_1 = nn.Dropout(drop_rate)

    self.linear2 = nn.Linear(hidden_dim,hidden_dim//2)
    self.bn2 = nn.BatchNorm1d(hidden_dim//2)
    self.drop_2 = nn.Dropout(drop_rate)

    self.fc_out = nn.Linear(hidden_dim//2, 1)  # Output 1 for binary classification

  def forward(self, features):
    x = F.relu(self.bn1(self.linear1(features))) # Use F.relu
    x = self.drop_1(x)

    x = F.relu(self.bn2(self.linear2(x))) # Use F.relu
    x = self.drop_2(x)

    x_out = torch.sigmoid(self.fc_out(x)) # Apply sigmoid for BCELoss

    return x_out

In [ ]:
model = MyNN(hidden_dim = 128)

epochs = 10

lr = 0.001

loss_function = nn.BCELoss()
from torch.optim import Adam
optim = Adam(params = model.parameters(), lr = lr)


In [ ]:
for epoch in range(epochs):
  for batch_loader, label_loader in train_data:
    optim.zero_grad()
    y_train_pred = model(batch_loader)
    loss = loss_function(y_train_pred,label_loader.view(-1, 1).float())
    loss.backward()
    optim.step()

In [ ]:
with torch.no_grad():
  pred_data = []
  actual_data = []
  for batch_loader, label_loader in test_data:
    outputs = model(batch_loader)
    outputs = (outputs > 0.5).int()
    pred_data.extend(outputs)
    actual_data.extend(label_loader)


from sklearn.metrics import accuracy_score

y_pred = np.array(pred_data).reshape(-1,1)
y_actual = np.array(actual_data).reshape(-1,1)

accuracy_score(y_actual,y_pred)

In [ ]:
with torch.no_grad():
  pred_data = []
  actual_data = []
  for batch_loader, label_loader in train_data:
    outputs = model(batch_loader)
    outputs = (outputs > 0.5).int()
    pred_data.extend(outputs)
    actual_data.extend(label_loader)


from sklearn.metrics import accuracy_score

y_pred = np.array(pred_data).reshape(-1,1)
y_actual = np.array(actual_data).reshape(-1,1)

accuracy_score(y_actual,y_pred)

In [ ]:
len(pred_data)